# Evaluation and optimization

## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Defining what "better" means

Before DSPy can improve our program automatically, we need to tell it what “better” means.

There are many ways to define your metric, but most use one or more of these patterns:

### 1. Labeled data comparisons

Compare the prediction against a “gold” output, usually labeled by a human expert. For example, a dataset of five-haiku arrays each paired with one ideal pick would let us evaluate and optimize the ensemble judge.
   ```text
   request 1: ..., gold answer 1: ...
   request 2: ..., gold answer 2: ...
   ...
   request n: ..., golder answer n: ...
   ```

### 2. Rule-based checks

Evaluate with code that checks verifiable properties. For our haiku program, this might involve:

- `check_lines(...):` (feedback: "haiku lines must be 3, got: 4")
- `check_syllables(...):` (feedback: "haiku must follow 5-7-5 rule for syllables, got: 5-6-6")
- ...etc.

### 3. [LLM judges](https://dspy.ai/diving-deeper/metrics-and-evaluation/)

Evaluate with another, usually larger, model. We might ask a frontier model how well a smaller model’s haiku evokes a place and season, similar to our ensemble’s judge. But here the optimizer bakes those judgments into the smaller model’s instructions once, instead of calling the judge with every inference.

## Preparing examples from the haiku dataset

We’ve created a dataset of example inputs by randomly grouping locations, seasons, and mood strings. [Click here to download the JSONL file](https://gist.github.com/dbreunig/b64412e6103d41889f3a87615008408d), containing 800 rows.

To prepare them for an evaluation or optimization, we need to convert each record into a `dspy.Example` object, like so:

In [4]:
import json

examples = []
with open("../datasets/haiku_examples.jsonl") as f:
    for line in f:
        row = json.loads(line)
        examples.append(
            dspy.Example(
                location=row["location"],
                season=row["season"],
                mood=row["mood"],
            ).with_inputs("location", "season", "mood")
        ) 

n = len(examples)
train_end = int(n * 0.75)
val_end = int(n * 0.875)
train, val, test = examples[:train_end], examples[train_end:val_end], examples[val_end:]

In [5]:
train[:3]

[Example({'location': 'Suburban garage with the garage door open', 'season': 'late winter', 'mood': 'frustrated'}) (input_keys={'mood', 'season', 'location'}),
 Example({'location': 'Saharan oasis', 'season': 'winter', 'mood': 'melancholy'}) (input_keys={'mood', 'season', 'location'}),
 Example({'location': 'Tokyo train platform', 'season': 'winter', 'mood': 'reverent'}) (input_keys={'mood', 'season', 'location'})]

In [6]:
print("splits:-")
print(len(train), len(val), len(test))

splits:-
600 100 100


The `.with_inputs` is how you tell DSPy which fields of an `Example` should be passed to the program at call time, as inputs.

A question-answering dataset is the more typical shape: each row carries `(question, answer)`, you’d call `.with_inputs(“question”)`, and DSPy would pass only the question to the program while holding `answer` back for the metric.

## Building our evaluation metric

A **Metric** is simply a Python function that scores a single prediction.

- A metric function for `dspy.Evaluate` accepts the original `example` and the program’s `prediction`, and returns a single float.
- By convention, scores fall in between 0.0 and 1.0 (higher is better).

We'll keep it super simple and only check if our season or mood inputs are being used verbatim in the haiku:

In [7]:
def haiku_score(example, prediction) -> float:
    """
    Penalize verbatim use of the input season string.
    A haiku should evoke the season through imagery, not name it
    directly.
    """
    text = prediction.haiku.lower()
    if example.season.strip().lower() in text:
        return 0.0
    return 1.0

Then we call `Evaluate`:

In [17]:
haiku_bot = dspy.Predict("location, season, mood -> haiku")

In [18]:
evaluate = dspy.Evaluate(devset=val, metric=haiku_score)
baseline_score = evaluate(haiku_bot)
print(baseline_score)

2026/06/17 10:55:01 INFO dspy.evaluate.evaluate: Average Metric: 55.0 / 100 (55.0%)


EvaluationResult(score=55.0, results=<list of 100 results>)


Our haiku writer, powered by `gpt-4.1-nano`, scores only 55% on this metric. And that makes sense. Our rule isn’t a hard and fast rule of haikus, so our model won’t adhere to it driven only by its weights. And our signatures gave no clues this was our objective.

See [Metrics and evaluation](https://dspy.ai/diving-deeper/metrics-and-evaluation/) for composite scoring, LLM judges, and test-set hygiene.

## Prompt Optimizing with GEPA

### Most teams start prompt-only and graduate to finetune only when prompt-only plateaus

> Prompt-only optimization costs LM tokens. Finetune costs LM tokens plus training compute plus deployment of new weights. The marginal lift of finetune over GEPA or MIPROv2 is usually small and sometimes negative, while the marginal cost is much larger. Treat finetune as the last lever, not the first. -- [Optimizers: choosing one](https://dspy.ai/diving-deeper/choosing-an-optimizer/#8-most-teams-start-prompt-only-and-graduate-to-finetune-only-when-prompt-only-plateaus)

Here is a [Cheat sheet](https://dspy.ai/diving-deeper/choosing-an-optimizer/#selection-cheat-sheet) for choosing an optimier from DSPy.

### Why we optimize prompts

- Different language models respond differently to the same prompt. (Gemini vs Claude)
- A request that gets clean structured output from one model can confuse another into rambling.
- Phrasing that works on today’s model can stop working when the provider ships an update next month. (GPT-5.3 vs GPT-5.4)
- Learning each model’s quirks by hand is slow, and the work doesn’t transfer.

And even if we stuck with only _one_ model, the potential permutations of our word choices and instructions are nearly infinite.

_Prompt optimization_ replaces that hand-tuning loop. We give DSPy a **training set** and a **metric**. It then generates variations of your instructions (using an LM), runs your examples with these candidate instructions, and keeps the highest-scoring prompts. We don’t have to know which prompt tricks a particular model responds to; **the optimizer finds them**.

The **savings** can be large. An optimized program on a smaller, cheaper model can often match a hand-prompted large model on the same task. **Reliability** improves as well, because the optimizer selects for what the metric rewards.

1. [Shopify converted a single-prompt GPT-5 task to DSPy](https://www.youtube.com/watch?v=bxToahwOVpY&t=1404s), moved to a small Qwen model, and optimized with GEPA, yielding a solution \~75x cheaper and \~2x more reliable.
2. [Dropbox used DSPy and prompt optimizers](https://dropbox.tech/machine-learning/optimizing-dropbox-dash-relevance-judge-with-dspy) to move to a smaller model and double program accuracy, labeling “10-100 times more data at the same cost.”

Better yet: when a new model launches next week, we can rerun the optimizer against it and immediately determine if it’s worth swapping in.

### GEPA uses reflection to improve instructions

DSPy ships with [several prompt optimizers](https://dspy.ai/diving-deeper/choosing-an-optimizer/), but today we’re going to focus on GEPA.

There are many reasons to like GEPA, but a key feature is it allows our metric to **provide text feedback** which the LM uses to inform subsequent instructions. Let’s update our original metric to demonstrate how this works:

In [ ]:
def haiku_score_gepa(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """
    Penalize verbatim use of the input season string.
    A haiku should evoke the season through imagery, not name it
    directly.
    """
    text = prediction.haiku.lower()
    if example.season.strip().lower() in text:
        return dspy.Prediction(
            score=0.0,
            feedback="Don't reference the input season verbatim."
        )
    return dspy.Prediction(score=1.0, feedback=None)

Instead of just returning a score, we can **tell the `reflection_lm` _why_ a prediction failed or succeeded**. Our example here is a bit silly, but this feedback ability is powerful. When labeling data to train against, labelers can jot down notes explaining a nuance in a specific record, which can then be passed to GEPA to guide future instructions. When training with an LM judge, the judge can provide detailed feedback for why a prediction didn’t meet the criteria.

### Expanding our haiku metric

To give our model more of a challenge, we’ve built out our metric to check for many conditions we expect from our haikus, specifically:

- Does it have the correct number of lines?
- Does it have the correct syllable count in each line?
- Does it avoid repeating inputs verbatim?
- Does it avoid the first-person voice?
- Does it have a balanced ratio of parts-of-speech?
- Does it use few adjectives?
- Does it use few articles?
- Does it use the present tense?

We can use the natural-language processing library [spaCy](https://spacy.io/) to assist with most of these measures.

All of these conditions make our metric too long to drop into this walkthrough, but [the code is available here](https://gist.github.com/dbreunig/228848f9b34bcdad6be37fc5f85ec1a0). Drop `haiku_metric.py` next to your notebook and add `from haiku_metric import haiku_metric` to follow along.

Installation:

In [ ]:
# ! uv add spacy
# ! pip install spacy

Download a trained pipeline (dependency for haiku_metric):


In [ ]:
# ! uv run python -m spacy download en_core_web_md

### Compiling our optimization

With our metric defined, it’s time to configure our optimizer:

In [ ]:
from haiku_metric import haiku_metric

reflection_lm = dspy.LM("openai/gpt-5.4")

optimizer = dspy.GEPA(
    metric=haiku_metric,

    # GEPA lets us choose a separate LM for reflection and instruction writing.
    # This LM looks at our examples and how they score, then rewrites our prompt in an attempt to improve our scores.
    # it’s worthwhile to use a larger model as the `reflection_lm`.
    # They’re better reasoners and prompters, and their cost isn’t a concern since they’re called a handful of times during optimization.
    reflection_lm=reflection_lm,

    # The `auto` argument sets our budget.
    # `auto="light"` evaluates around six candidate prompts before stopping.
    # `"medium"` and `"heavy"` options go further, and our GEPA deep dive covers additional levers we can set.
    auto="light",

    # Depending on your inference provider you may have to tweak this to avoid any rate limits.
    num_threads=2,
)

[  0.0s] importing spacy...
[  0.5s] importing dspy...
[  0.5s] loading model...
[  2.5s] ready.


Finally, we compile our optimized program:

In [22]:
optimized_haiku_bot = optimizer.compile(haiku_bot, trainset=train, valset=val)

2026/06/17 14:18:06 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 780 metric calls of the program. This amounts to 1.11 full evals on the train+val set.
2026/06/17 14:18:06 INFO dspy.teleprompt.gepa.gepa: Using 100 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/780 [00:00<?, ?rollouts/s]2026/06/17 14:18:10 INFO dspy.evaluate.evaluate: Average Metric: 79.86350702633472 / 100 (79.9%)
2026/06/17 14:18:10 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.7986350702633467 over 100 / 100 examples
GEPA Optimization:  13%|█▎        | 100/780 [00:04<00:27, 24.70rollouts/s]2026/06/17 14:18:10 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Select

Average Metric: 2.33 / 3 (77.6%): 100%|██████████| 3/3 [00:04<00:00,  1.51s/it]

2026/06/17 14:18:15 INFO dspy.evaluate.evaluate: Average Metric: 2.3269422314564387 / 3 (77.6%)


2026/06/17 14:18:24 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for self: Given the input fields `location`, `season`, and `mood`, generate exactly one field: `haiku`.

Requirements for `haiku`:
- Output only the haiku text, formatted as exactly 3 lines.
- Use a strict 5–7–5 syllable structure, one line per count.
- The poem should evoke the given `location`, reflect the given `mood`, and include a concrete seasonal reference tied to the given `season`.
- Favor classical haiku qualities:
  - concrete sensory imagery over abstraction
  - sparse adjectives
  - strong use of nouns and present-tense verbs
  - no first-person pronouns
  - one compact sentence at most
  - low use of articles and other stop words
  - high lexical density
- Include a clear juxtaposition or “cut” by ending line 1 or line 2 with an em dash (`—`), colon (`:`), or ellipsis (`...`).
- Make line 1 and line 3 meaningfully distinct to strengthen juxtaposition.
- Avoid repeating content words or lemm

Average Metric: 2.63 / 3 (87.7%): 100%|██████████| 3/3 [00:02<00:00,  1.41it/s]

...

2026/06/17 14:25:19 INFO dspy.evaluate.evaluate: Average Metric: 2.4719610950712 / 3 (82.4%)


2026/06/17 14:25:30 INFO dspy.teleprompt.gepa.gepa: Iteration 8: Proposed new text for self: You will receive exactly three input fields:

- `location`
- `season`
- `mood`

Your job is to output only a single haiku, with no labels or extra text.

OUTPUT RULES
- Return only the haiku text.
- Output exactly 3 lines.
- Do not include titles, labels, JSON, bullets, explanations, quotes, or commentary.
- Use at most one sentence across the full haiku.

HARD CONSTRAINTS
- The haiku must be exact 5–7–5 syllables, line by line.
- Recount syllables conservatively before answering. If a word’s count is doubtful, replace it.
- End line 1 or line 2 with exactly one cut mark: `—`, `:` or `...`
- Keep syntax simple, shallow, and direct.
- Do not use first-person pronouns.
- Avoid repeated content words or close lemmas across lines.

PRIMARY GOALS
1. Root the haiku in the given `location` using specific, plausible, grounded imagery.
2. Make the given `season` explicit through a concrete seasonal mark

Average Metric: 2.58 / 3 (86.1%): 100%|██████████| 3/3 [00:01<00:00,  1.52it/s]

2026/06/17 14:26:35 INFO dspy.evaluate.evaluate: Average Metric: 2.582820382459468 / 3 (86.1%)


2026/06/17 14:26:46 INFO dspy.teleprompt.gepa.gepa: Iteration 9: Proposed new text for self: You are given exactly three input fields:
- `location`
- `season`
- `mood`

Your job is to generate exactly one output field:
- `haiku`

Output contract:
- Return only the value for `haiku`.
- Do not add labels, JSON, prose, commentary, explanations, or multiple options.
- The haiku must be exactly 3 lines.

Core form requirements:
- Use a strict 5–7–5 syllable pattern, one line per count.
- Silently verify syllables before finalizing; if uncertain, prefer simpler, highly countable words.
- Keep to one compact sentence at most.
- Include a clear cut by ending line 1 or line 2 with exactly one of these marks: `—` or `:` or `...`.

Content requirements:
- Root the poem specifically in the given `location`; avoid generic nature imagery if the location offers concrete details.
- Reflect the given `mood` indirectly through scene, tension, pacing, temperature, sound, distance, emptiness, obstruction,

```mermaid
flowchart TB
    subgraph outer["GEPA optimization loop (auto='light' ≈ 6 candidate prompts)"]
        direction TB

        START([optimizer.compile<br/>train=600, val=100]) --> CAND[Build / mutate candidate prompt<br/>instructions for haiku_bot]
        CAND --> EVAL_PASS[Evaluation pass over a batch<br/>val set and/or reflection minibatch]

        subgraph parallel["ParallelExecutor — num_threads=2"]
            direction LR

            subgraph T1["Thread 1"]
                E1["① Pick dspy.Example<br/>(location, season, mood)"]
                E2["② Run student program<br/>(compiled haiku_bot)"]
                E3["③ LLM API call<br/>OpenRouter / gpt-4.1-nano"]
                E4["④ Run haiku_metric locally<br/>(spaCy checks, feedback)"]
                E5["⑤ Return score + trace"]
                E1 --> E2 --> E3 --> E4 --> E5
            end

            subgraph T2["Thread 2"]
                F1["① Pick next Example"]
                F2["② Run student program"]
                F3["③ LLM API call"]
                F4["④ Run haiku_metric"]
                F5["⑤ Return score + trace"]
                F1 --> F2 --> F3 --> F4 --> F5
            end
        end

        EVAL_PASS --> parallel
        E5 --> AGG
        F5 --> AGG

        AGG["⑤ Aggregate EvaluationBatch<br/>100 val scores per full pass<br/>+ metric feedback text"]
        AGG --> DECIDE{Budget left?<br/>more candidates?}

        DECIDE -->|yes| REFLECT["reflection_lm (gpt-5.4)<br/>reads failures + feedback<br/>rewrites instructions"]
        REFLECT --> CAND

        DECIDE -->|no| BEST["Return best candidate<br/>highest val aggregate score"]
    end
```

### Inspecting post-tuning performance

Let's compare it to a bigger model (unoptimized):

In [ ]:
haiku_bot_big_unoptimized = dspy.Predict("location, season, mood -> haiku")
haiku_bot_big_unoptimized.set_lm(lm=dspy.LM("openrouter/openai/gpt-5.4-nano"))

In [ ]:
evaluate2 = dspy.Evaluate(devset=val, metric=haiku_score)
big_unoptimized_score = evaluate2(haiku_bot_big_unoptimized)
print(big_unoptimized_score)

2026/06/17 15:05:39 INFO dspy.evaluate.evaluate: Average Metric: 13.0 / 100 (13.0%)


EvaluationResult(score=13.0, results=<list of 100 results>)


Compare scoring before and after prompt-tuning:

- $ 55 \% $ (`gpt-4.1-nano`)
- $ 88 \% $ (`gpt-4.1-nano-optimized`)
- $ 13 \% $ (`gpt-5.4-nano`)

Surprisingly, the `4.1` to `5.4` did better even before optimization! Things like these aren't obvious until measured.

### Looking at the refined prompt

Our program’s synthesis step started off with these instructions, which we defined in our signature docstring:

In [ ]:
print(haiku_bot.signature.instructions)

Given the fields `location`, `season`, `mood`, produce the fields `haiku`.


After compile, these became:

In [ ]:
print(optimized_haiku_bot.signature.instructions)

You will receive exactly three input fields:

- `location`
- `season`
- `mood`

Your job is to output only a single haiku, with no labels or extra text.

OUTPUT RULES
- Return only the haiku text.
- Output exactly 3 lines.
- Do not include titles, labels, JSON, bullets, explanations, quotes, or commentary.
- Use at most one sentence across the full haiku.

HARD CONSTRAINTS
- The haiku must be exact 5–7–5 syllables, line by line.
- Recount syllables conservatively before answering. If a word’s count is doubtful, replace it.
- End line 1 or line 2 with exactly one cut mark: `—`, `:` or `...`
- Keep syntax simple, shallow, and direct.
- Do not use first-person pronouns.
- Avoid repeated content words or close lemmas across lines.

PRIMARY GOALS
1. Root the haiku in the given `location` using specific, plausible, grounded imagery.
2. Make the given `season` explicit through a concrete seasonal marker, not merely a mood word and not only the season name.
3. Convey the given `mood` indirectl

Fascinatingly, the same program will optimize differently depending on the model.

See [GEPA in depth](https://dspy.ai/diving-deeper/gepa-in-depth/) for the full mechanics: Pareto sampling, per-predictor feedback, `auto` budget translation, and the `detailed_results` audit trail.

## Saving a program and reloading it

### A. Saving the optimized state

Let’s quickly save our optimized program, then take a look at how our prompt changed.

In [ ]:
# state-only — small file, requires re-instantiating the program before loading
optimized_haiku_bot.save("react_gpt_nano_haiku_optimized.json")

### B. Saving the entire program

Reach for `save_program=True` when whoever is loading the program won’t have your Python class definitions handy.

1. Shipping an optimized program to another team is one example
2. serving it from a different repo than the one you developed it in is another

The cost is that the saved directory contains executable Python, so only load programs from sources you trust.

In [47]:
# whole program — directory, rehydrates without you re-defining the class
optimized_haiku_bot.save("react_gpt_nano_haiku_optimized/", save_program=True)

2026/06/17 14:46:13 WARNING dspy.primitives.base_module: Loading untrusted .pkl files can run arbitrary code, which may be dangerous. To avoid this, prefer saving using json format using module.save("module.json").


### Reloading a saved program

For the directory form, `dspy.load(path)` rehydrates the full module in one call. For the state-only form, you build a fresh copy of the program in code and call `.load(path)` on it to apply the saved state.

In [49]:
# whole program
loaded = dspy.load("react_gpt_nano_haiku_optimized/", allow_pickle=True)
loaded

Predict(StringSignature(location, season, mood -> haiku
    instructions='You will receive exactly three input fields:\n\n- `location`\n- `season`\n- `mood`\n\nYour job is to output only a single haiku, with no labels or extra text.\n\nOUTPUT RULES\n- Return only the haiku text.\n- Output exactly 3 lines.\n- Do not include titles, labels, JSON, bullets, explanations, quotes, or commentary.\n- Use at most one sentence across the full haiku.\n\nHARD CONSTRAINTS\n- The haiku must be exact 5–7–5 syllables, line by line.\n- Recount syllables conservatively before answering. If a word’s count is doubtful, replace it.\n- End line 1 or line 2 with exactly one cut mark: `—`, `:` or `...`\n- Keep syntax simple, shallow, and direct.\n- Do not use first-person pronouns.\n- Avoid repeated content words or close lemmas across lines.\n\nPRIMARY GOALS\n1. Root the haiku in the given `location` using specific, plausible, grounded imagery.\n2. Make the given `season` explicit through a concrete seasonal

In [50]:
# state-only
fresh = dspy.Predict("location, season, mood -> haiku")
fresh.load("react_gpt_nano_haiku_optimized.json")
fresh

Predict(StringSignature(location, season, mood -> haiku
    instructions='You will receive exactly three input fields:\n\n- `location`\n- `season`\n- `mood`\n\nYour job is to output only a single haiku, with no labels or extra text.\n\nOUTPUT RULES\n- Return only the haiku text.\n- Output exactly 3 lines.\n- Do not include titles, labels, JSON, bullets, explanations, quotes, or commentary.\n- Use at most one sentence across the full haiku.\n\nHARD CONSTRAINTS\n- The haiku must be exact 5–7–5 syllables, line by line.\n- Recount syllables conservatively before answering. If a word’s count is doubtful, replace it.\n- End line 1 or line 2 with exactly one cut mark: `—`, `:` or `...`\n- Keep syntax simple, shallow, and direct.\n- Do not use first-person pronouns.\n- Avoid repeated content words or close lemmas across lines.\n\nPRIMARY GOALS\n1. Root the haiku in the given `location` using specific, plausible, grounded imagery.\n2. Make the given `season` explicit through a concrete seasonal

### What does the saved file contain?

The save file contains the optimized instructions, demos, and signature metadata. It does not contain the LM client configuration: your API keys, your provider choice, your temperature. That separation is intentional: configure your LM as usual after loading and the same program targets whichever model you point it at today.

See [Saving and loading](https://dspy.ai/diving-deeper/saving-and-loading/) for versioning saved programs, swapping models against the same checkpoint, and managing demo lifecycles.

## Where to go Next?

See [Where to go next](https://dspy.ai/getting-started/where-to-go-next/).